# Phase 6.5C.1 — Held-out cohort preparation (zero LLM calls)
Frozen candidate agent: 67cc048b69369f1fd12b1eccd131c454e762a71a. Three public SAPs, nine purposively selected table targets. Candidate cohort only until study provenance and independent reference labels are reviewed. Downloads from ClinicalTrials.gov go directly to Drive. Reference pages are excluded from inference. No previous ZIPs or DeepSeek secret needed. Do not tune on this cohort; if a substantive algorithm change is needed, retire it to development and select a new held-out cohort.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, json, shutil, time, re, hashlib
from datetime import datetime, timezone
REPO='https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH='simplified-rag-v3'
PIN='67cc048b69369f1fd12b1eccd131c454e762a71a'
ROOT=Path('/content/tfl-phase65c-frozen')
if not ROOT.exists():
    subprocess.run(['git','clone','--branch',BRANCH,REPO,str(ROOT)],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach',PIN],check=True)
assert subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()==PIN
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC=ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in (SRC/'tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
from tfl_agent.agent.unified_pipeline import TFLAgent, ResponseExecutor, MATCH_SYSTEM, sha_file, write_json, digest
from tfl_agent.agent.phase6_simplified_replay import APP, DISC
assert all('"'+x+'"' in MATCH_SYSTEM for x in APP|DISC|{'supported','uncertain','incompatible','sap','title'})
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests/test_unified_pipeline.py')],check=True)
print('Frozen commit:',PIN)

In [ ]:
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive',timeout_ms=180000)
BASE=Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
assert BASE.is_dir()
LIBRARY=ROOT/'generic_template_library'
DATA=BASE/'Phase65C_Heldout_SAPs'
DEST=BASE/'Phase65C_Preparation_Outputs'
DATA.mkdir(parents=True,exist_ok=True)
DEST.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=Path('/content/Phase65C_Preparation')/RUN_ID
OUT.mkdir(parents=True,exist_ok=False)
# Page coordinates are physical, 1-based, inclusive, verified visually against public versions.
# Exclude the entire appended shell section, even its list of tables.
STUDIES=[
 {'study_id':'RV546','nct':'NCT04658667','body_pages':[8,26],'boundary_page':28,
  'boundary_marker':'APPENDIX 1. TABLE SHELLS','identity_marker':'RV546',
  'targets':[
   ['5','Participant Disposition by Vaccination Group'],
   ['14','Adverse Events Occurring in 5% of Participants in Any Vaccination Group by MedDRA System Organ Class and Preferred Term, and Vaccination Group'],
   ['16','Number and Percentage of Participants Experiencing Solicited Reactogenicity Events with 95% Confidence Intervals, by Event and Vaccination Group']]},
 {'study_id':'RESP30X-EBA','nct':'NCT06041919','body_pages':[1,36],'boundary_page':37,
  'boundary_marker':'12.1 Table Shells','identity_marker':'RESP30X-EBA',
  'targets':[
   ['14.1.2','Demographic Data'],
   ['14.1.3','Medical History'],
   ['14.3.2.1.2','Clinical Safety Laboratory Tests – Haematology Change from Baseline over time']]},
 {'study_id':'B3461058','nct':'NCT03842163','body_pages':[1,17],'boundary_page':18,
  'boundary_marker':'LIST OF TABLES AND TABLE SHELLS','identity_marker':'B3461058',
  'targets':[
   ['15.1.1','Number and Percentage of Patients in Each Analysis Set'],
   ['15.1.2-FAS1-2','Demography and Baseline Characteristics – FAS1, 2'],
   ['15.2.1','Prevalence of Patients with Cardiac Fixation at the Radionuclide Bone Scintigraphy and / or SPECT Performed – FAS1']]}
]
registry=json.loads((ROOT/'library/Library_Registry.json').read_text())
excluded={x['study_id'].casefold() for x in registry['studies']}
assert all(s['study_id'].casefold() not in excluded for s in STUDIES)
print('Excluded Library source studies:',sorted(excluded))
print('Selected targets:',sum(len(s['targets']) for s in STUDIES))
print('New LLM attempts: 0')

## Download and verify page boundaries
No manual upload or page-number entry. If a public file changes or a marker does not match, stop rather than guess. The original PDFs are kept for independent annotation; only the stated body page ranges may enter Agent preparation. Public availability does not itself establish permission to send content to a third-party model. This notebook sends no excerpts externally.

In [ ]:
import requests, pymupdf
def norm(s):
    return re.sub(r'\s+',' ',s).strip().casefold()
manifest=[]
for s in STUDIES:
    nct=s['nct']
    url=f'https://cdn.clinicaltrials.gov/large-docs/{nct[-2:]}/{nct}/SAP_001.pdf'
    path=DATA/f'{nct}_SAP_001.pdf'
    if not path.exists():
        response=requests.get(url,timeout=120)
        response.raise_for_status()
        assert response.content.startswith(b'%PDF-'), 'Not a PDF'
        tmp=path.with_suffix('.tmp')
        tmp.write_bytes(response.content)
        tmp.replace(path)
    with pymupdf.open(path) as doc:
        first,last=s['body_pages']
        assert 1<=first<=last<s['boundary_page']<=len(doc)
        head=' '.join(doc[i].get_text() for i in range(min(3,len(doc))))
        assert norm(s['identity_marker']) in norm(head), f'Identity mismatch: {nct}'
        boundary=doc[s['boundary_page']-1].get_text()
        assert norm(s['boundary_marker']) in norm(boundary), f'Boundary mismatch: {nct}'
        assert all(norm(s['boundary_marker']) not in norm(doc[i-1].get_text()) for i in range(max(first,last-1),last+1)), f'Boundary marker found at end of body: {nct}'
        record={**s,'url':url,'relative_path':str(path.relative_to(BASE)),
                'sap_sha256':sha_file(path),'pdf_pages':len(doc),
                'body_last_excerpt':doc[last-1].get_text()[-500:],
                'boundary_excerpt':boundary[:500]}
        manifest.append(record)
        print(nct,'verified',len(doc),'pages; body',first,last,'SHA256',record['sap_sha256'])
write_json(OUT/'Cohort_Manifest.json',manifest)
write_json(OUT/'Freeze.json',{'commit':PIN,'matching_prompt_sha256':digest(MATCH_SYSTEM),
                             'source_library_studies':sorted(excluded),'heldout_status':'candidate_pending_provenance_and_labels',
                             'new_attempts':0,'selection_policy':'purposive 3 targets per study, chosen before inference'})
# Independent reviewer sheet contains only case identity, not model retrieval or predictions.
labels=[]
for s in manifest:
    for entry,title in s['targets']:
        labels.append({'case_id':s['study_id']+'__'+entry,'study_id':s['study_id'],'target_title':title,
                       'sap_sha256':s['sap_sha256'],'body_pages':s['body_pages'],
                       'expected_template_ids':None,'reference_shell_pages':None,
                       'gold_evidence':None,'gold_claims':None,'reviewer':None,'status':'unreviewed'})
# Keep reference labels separate from inference plans; never pass them to TFLAgent.
write_json(OUT/'Independent_Labels_Blank.json',labels)
print('Independent labels must be completed before paid held-out inference.')

## Zero-call retrieval preparation
The agent uses a single local encoder and reuses the SAP index for targets of the same PDF. Save full request plans, source-span provenance, and complete normalized chunk text for independent retrieval scoring. No model response or template prediction is produced. A failed case stays in the cohort; no replacement after seeing outputs.

In [ ]:
import torch
from tfl_agent.tools.sap_hybrid import LocalEncoder
executor=ResponseExecutor()
agent=TFLAgent(LIBRARY,encoder=LocalEncoder(device='cuda' if torch.cuda.is_available() else 'cpu'),executor=executor)
rows=[]
for s in manifest:
    path=BASE/s['relative_path']
    assert sha_file(path)==s['sap_sha256']
    for entry,title in s['targets']:
        cid=s['study_id']+'__'+entry
        start=time.perf_counter()
        row={'case_id':cid,'study_id':s['study_id'],'target':title}
        try:
            plan=agent.prepare(path,title,body_pages=tuple(s['body_pages']),study_id=s['study_id'],entry_id=entry)
            first,last=s['body_pages']
            assert plan['sap_sha256']==s['sap_sha256']
            assert all(first<=p<=last for c in plan['source_chunks'] for p in c['pages'])
            assert set(plan['specification_request']['config']['payload'])=={'target','allowed_fields','sap_evidence'}
            assert plan['matching_request']['config']['system']==MATCH_SYSTEM
            write_json(OUT/'Plans'/f'{cid}.json',plan)
            # Full normalized corpus supports later mapping independent gold evidence to all chunks.
            corpus=agent._study_cache[(s['sap_sha256'],tuple(s['body_pages']))][0]
            assert all(first<=p<=last for c in corpus for p in c['pages'])
            corpus_path=OUT/'Corpus'/f"{s['study_id']}.json"
            if not corpus_path.exists():
                write_json(corpus_path,corpus)
            row.update(status='prepared',blocks=plan['retrieval']['block_count'],chunks=len(corpus),
                       primary=len(plan['retrieval']['primary_ids']),supplementary=len(plan['retrieval']['supplementary_ids']),
                       matching_fingerprint=plan['matching_request']['fingerprint'],
                       specification_fingerprint=plan['specification_request']['fingerprint'])
        except Exception as exc:
            row.update(status='failed',error_type=type(exc).__name__)
            print(cid,type(exc).__name__,str(exc)[:300])
        row['seconds']=round(time.perf_counter()-start,3)
        rows.append(row)
        print(cid,row['status'],row.get('primary'),row.get('supplementary'))
assert executor.new_attempts==0
summary={'commit':PIN,'cases':len(rows),'prepared':sum(r['status']=='prepared' for r in rows),
         'new_attempts':0,'paid_budget_requested_next':2*len(rows),'heldout_accuracy':None,
         'cohort_status':'candidate_pending_independent_labels','rows':rows}
write_json(OUT/'Summary.json',summary)
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print(json.dumps(summary,indent=2,ensure_ascii=False))
print('Share preparation ZIP only if permitted; it contains SAP excerpts. Do not share secrets.')

## Independent evaluation protocol
Before live inference: verify no study was used in previous development beyond the checked Library registry; freeze the cohort and label each target from the reference shell and the SAP body without inspecting retrieval rankings or model outputs. Record acceptable template IDs or explicit out-of-library disposition, reference pages, target-specific supporting SAP passages with physical page numbers, atomic field/value claims and whether they are inferable from the body/title or shell-only. Separate shell-only requirements from extraction recall denominator. For retrieval, map each independent evidence unit to source chunk spans (including cross-chunk evidence); distinguish corpus coverage, primary recall, augmented recall, and chunk relevance precision. For live evaluation record top-1/top-3, abstention, protocol validity, raw vs retained semantic claim precision/recall, alignment repairs and regressions, token/latency, failures in denominators. No score before labels; no paid calls without explicit cap authorization. Nine cases would require at most eighteen completion attempts. This is a small purposive cohort, not population-representative.